# JetBot line follower — v18

Built from v13 (CJ). Ordinary line following and gap crossing are **unchanged** (verified frame-by-frame against v13).

| Problem | Cause | Change | Knob |
|---|---|---|---|
| v18: a green turn sometimes keeps rotating (into a U-turn) | The turn ended only when the line was centred in near **and** middle for 3 frames. If the axle isn't exactly over the junction, the line sweeps through the view diagonally and never meets both limits, so the robot spins past it. | The turn also ends when the new line, first seen ahead on the turning side, crosses the centre (2 frames). A line seen well past the centre resets this, so stray strokes can't end a U-turn early. | `GREEN_AHEAD_MIN`, `GREEN_CROSS_TOL`, `GREEN_CROSS_FRAMES` |
| v17: second junction right after a green turn is not taken | Its marker is only visible while the first turn is still running, and votes were ignored during turns. By the time the turn and cooldown ended, the marker was under the robot. | Valid markers seen in the second half of a turn are remembered. If 2 agree when the turn ends, the next maneuver starts immediately, with no cooldown. | `GREEN_CHAIN_START`, `JUNCTION_REACQUIRE` |
| v16: marker approached at an angle (e.g. off a zig-zag) reads `CENTER` or the wrong side | The crossbar looks diagonal in the image, so the row test found no crossbar. Left/right was judged against the near→far band points, which jump onto the branch. | Crossbars are also searched on the mask rotated by ±15°/±30°. Left/right is judged against the approach line traced up from the bottom to the marker's height. A dead-end pair is judged together even if one marker reaches the zone first. | `JUNCTION_ANGLES`, `GREEN_TRACE_*` |
| v15: green marker right after a turn (green or sharp black) is seen but not acted on | A marker only counted if the **lowest** horizontal stroke in view was above it. Right after a turn, the line just left lies across the bottom of the image, so it was the lowest stroke and every marker was rejected (`marker=CENTER`). | Each marker is matched to the **nearest crossbar above it**. The advance then follows that crossbar, so the old line can't trigger an early turn. | `GREEN_JUNCTION_MAX_GAP` |
| v15: finish stops too early | Red then silver anywhere in the lower half of the image stopped the robot. | The robot stops only when the foil's lowest edge reaches the near band. An optional roll time is available after that. | `FINISH_STOP_Y`, `FINISH_ROLL_TIME` |
| v14: back-to-back markers missed | 0.4 s blind cooldown plus a 3-of-5 frame vote | 2 votes within 0.30 s; cooldown 0.15 s | `GREEN_VOTES_NEEDED`, `GREEN_COOLDOWN` |
| v14: advance overshoots | 0.12 s extra straight driving, then forward roll in the turn ramp | Extra driving 0.04 s | `GREEN_POST_JUNCTION_TIME` |

**Layout:** 1 Setup → 2 Parameters (every tuning value) → 3 Perception → 4 Controllers → 5 Preview and drive.
**Preview overlay:** thin red = every crossbar, drawn at its tilt; yellow dot = where the green marker's crossbar meets the line.
The status line's `gate` shows why a seen marker isn't acted on yet (`cooldown`, `blocked`, `votes 1/2`); during a turn, `chain 1/2` counts markers remembered for the next junction.

## 1. Setup

In [ ]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()
camera = Camera.instance(width=224, height=224)
live_view = widgets.Image(format="jpeg", width=336, height=336)
camera_link = traitlets.dlink((camera, "value"), (live_view, "value"), transform=bgr8_to_jpeg)
display(live_view)

latest = {"sample": None}  # newest frame only: decisions are made on fresh data, never on a queue

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")
time.sleep(0.5)
print("Camera frame received." if latest["sample"] is not None else "No camera frame yet.")

## 2. Parameters
Every tuning value in one place, grouped by subsystem. Lines marked **v14** – **v18** are the ones changed in those versions.

In [ ]:
# All tuning values live in this cell. Re-run it after changing anything (no other cell needs re-running,
# except that controller objects are rebuilt at the start of every drive() call anyway).

# ---------------------------------------------------------------- motors
BASE_SPEED = 0.12
CURVE_SPEED = 0.12
MIN_DRIVE_SPEED = 0.12      # lowest command that still reliably moves the robot
MAX_SPEED = 0.25
KP = 0.14
TURN_SPEED = 0.15           # black-line pivot, outer wheel
GREEN_TURN_SPEED = 0.15     # green turn, both wheels (one reversed)
GREEN_TURN_SLOW_SPEED = 0.10  # lowest speed that still reliably rotates in place
PIVOT_INNER_RATIO = -0.6    # inner wheel reverses at 60 % -> rotation near the axle

# ---------------------------------------------------------------- measured geometry / timing
ROTATION_TIME_360 = 2.15    # CALIBRATE with spin_test(): seconds per full in-place turn at GREEN_TURN_SPEED
FRAME_TIMEOUT = 0.5         # stop if the newest camera frame is older than this

# Derived
QUARTER_TURN_TIME = ROTATION_TIME_360 / 4.0
# Black-line pivot rotates slower: wheel-speed difference is (1 - ratio)*TURN_SPEED instead of 2*GREEN_TURN_SPEED
LINE_PIVOT_TIME_360 = ROTATION_TIME_360 * (2.0 * GREEN_TURN_SPEED) / ((1.0 - PIVOT_INNER_RATIO) * TURN_SPEED)

# ---------------------------------------------------------------- perception: black line
BLACK_THRESHOLD = 70
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]
CROSS_BAND_WIDTH = 0.40     # contour wider than this fraction of the image = horizontal stroke in the band
STROKE_FILL = 0.85          # a column black over this fraction of the band height belongs to the through-line
CROSS_ROW_FRACTION = 0.25   # a row with more black than this fraction of width belongs to a horizontal stroke
CROSS_MIN_ROWS = 3
CROSS_SEARCH = (0.30, 0.95)
# v16: crossbars seen at an angle (robot not square to the junction) are found by repeating the row test
# on the mask rotated by these angles (degrees). Larger angles risk reading the robot's own line as a crossbar.
JUNCTION_ANGLES = (0, -15, 15, -30, 30)
CROSS_MAX_ROWS_TILTED = 0.20  # a tilted "stroke" thicker than this fraction of height is the line itself, not a crossbar
JUNCTION_MERGE = 0.05         # strokes closer than this (fraction of height) are the same crossbar

# ---------------------------------------------------------------- perception: green markers
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0
GREEN_PAIR_Y_TOLERANCE = 0.15
# v15: a marker belongs to the nearest crossbar ABOVE it, which must be within this fraction of image height.
# (v13 used the lowest stroke in view, so the old line lying across the bottom after a turn hid every marker.)
GREEN_JUNCTION_MAX_GAP = 0.22
# v16: left/right is judged against the approach line traced up from the bottom of the image
GREEN_TRACE_START = 0.92      # row where the trace starts
GREEN_TRACE_MAX_WIDTH = 0.30  # wider black runs are crossbars, not the line
GREEN_TRACE_MAX_STEP = 8      # px the line may shift between traced rows (every 2nd row)
GREEN_TRACE_MAX_MISSES = 8    # rows without the line before giving up
GREEN_TRACE_MARGIN = 10       # px: trace up to just above the highest marker

# ---------------------------------------------------------------- perception: finish (red band, then silver foil)
FINISH_RED_MIN_COVERAGE = 0.30
FINISH_SILVER_MIN_COVERAGE = 0.48
FINISH_ROW_MIN_PIXELS = 3
FINISH_ROI_TOP = 0.42
FINISH_ROI_BOTTOM = 0.96
FINISH_SEQUENCE_TIMEOUT = 3.0
# v15: stop only when the foil reaches the robot: its lowest edge must be at or below this image row
# (near band is 0.72-0.88). Then drive straight FINISH_ROLL_TIME more (0 = stop at once) if the robot
# should end further onto the foil.
FINISH_STOP_Y = 0.80
FINISH_ROLL_TIME = 0.0

# ---------------------------------------------------------------- black-line controller
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_EXIT_MIDDLE = 0.35     # middle band need only be roughly aligned; the next bend may already be visible
LOOKAHEAD_GAIN = 0.40
ZIGZAG_SPREAD = 0.60        # near and far on opposite sides by this much = two bends in view
STEERING_FILTER_ALPHA = 0.55  # light smoothing; lower is smoother but slower to react
BAND_WEIGHTS = {"near": 0.45, "middle": 0.35, "far": 0.20}
TURN_PROGRESS = 0.03
TURN_STALL_TIMEOUT = 0.8    # fail only if the error stops shrinking for this long
TURN_HARD_TIMEOUT = 0.5 * LINE_PIVOT_TIME_360   # never rotate more than ~180 degrees
SEARCH_TIMEOUT = LINE_PIVOT_TIME_360 / 3.0      # search up to ~120 degrees

LINE_CENTER_LIMIT = 0.22         # near + middle centred -> ignore the far band (side branches)
LINE_MIDDLE_CENTER_LIMIT = 0.35
LINE_TURN_NEAR_MIN = 0.40        # a pivot needs near AND middle on the same side...
LINE_TURN_MIDDLE_MIN = 0.20
LINE_TURN_CONFIRM_FRAMES = 3     # ...for this many frames
LINE_TURN_NEAR_ONLY_MIN = 0.70   # corner where the middle band is empty
LINE_TURN_NEAR_ONLY_FRAMES = 5
LINE_PENDING_ERROR_LIMIT = 0.30  # steering limit while a pivot is not yet confirmed

# ---------------------------------------------------------------- gap crossing
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 2.3
GAP_STEER_LIMIT = 0.015
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

# ---------------------------------------------------------------- green markers: confirmation
# v14: votes are counted over a short TIME window instead of "3 of the last 5 frames", and are not thrown
# away while the line controller is briefly busy. A marker seen right after a previous turn is only in the
# trigger zone for a few frames, which the old 3-of-5 rule plus a 0.4 s blind cooldown could not catch.
GREEN_VOTE_WINDOW_TIME = 0.30   # seconds of history that count
GREEN_VOTES_NEEDED = 2          # frames of the same direction inside that window
GREEN_COOLDOWN = 0.15           # after a green turn, ignore markers this long (was 0.4)
GREEN_BOTH_UPGRADE = 2          # BOTH frames seen early in the advance that turn LEFT/RIGHT into a U-turn
GREEN_RECHECK_TIME = 0.20       # how long into the advance the BOTH upgrade is allowed
# v17: when two junctions are very close, the next marker is only visible while the current turn is still
# running. Valid markers seen after this much of the turn are remembered; if they confirm a direction when the
# turn ends, the next maneuver starts immediately (no cooldown).
GREEN_CHAIN_START = 0.6         # fraction of the expected turn time after which markers are remembered
JUNCTION_REACQUIRE = 0.15       # chained advance: how far the crossbar may have moved while the robot rotated

# ---------------------------------------------------------------- green markers: advance to the junction
GREEN_ADVANCE_SPEED = 0.12
GREEN_ADVANCE_STEER = 0.02
JUNCTION_NEAR_Y = 0.75          # crossbar must have come this low before its disappearance counts
JUNCTION_EXIT_Y = 0.90          # ...or reached this row
# v15: the advance follows the marker's own crossbar, ignoring other strokes (e.g. the line just turned off)
JUNCTION_TRACK_BACK = 0.05      # tracked crossbar may appear to move up this much (noise)
JUNCTION_TRACK_STEP = 0.15      # ...or down this much between frames
GREEN_PASS_FRAMES = 2           # frames the crossbar must read as "under the camera"
GREEN_NO_JUNCTION_WAIT = 0.30   # crossbar never seen while advancing: assume already under the camera
GREEN_ADVANCE_TIMEOUT = 2.0
# v14: extra straight driving after the crossbar leaves the image (was CAMERA_BLIND_TIME = 0.12).
# The turn ramp below still rolls forward for about half of GREEN_TURN_RAMP_TIME, and the robot no longer
# stops first, so the old value overshot. Overshoots -> lower it; pivots short of the junction -> raise it.
GREEN_POST_JUNCTION_TIME = 0.04

# ---------------------------------------------------------------- green markers: the turn itself
GREEN_TURN_RAMP_TIME = 0.12     # blend from forward driving into the in-place turn
GREEN_TURN_MIN_TIME = 0.6 * QUARTER_TURN_TIME
GREEN_TURN_TIMEOUT = 2.5 * QUARTER_TURN_TIME
UTURN_MIN_TIME = 0.75 * 2 * QUARTER_TURN_TIME
UTURN_TIMEOUT = 2.0 * 2 * QUARTER_TURN_TIME
GREEN_REACQUIRE_FRAMES = 3      # line centred for this many frames = turn finished
# v18: if the axle is not exactly over the junction, the new line sweeps through the view diagonally and is
# never "centred" in near AND middle at once, so the robot spun past it. The turn now also ends when the new
# line, first seen ahead of the centre in the turning direction, has crossed the centre.
GREEN_AHEAD_MIN = 0.25          # line this far ahead (turning side) = the new line is approaching
GREEN_CROSS_TOL = 0.05          # ...and is "crossed" once it is within this of the centre or past it
GREEN_CROSS_FRAMES = 2          # frames the crossing must hold (rejects a single noisy frame)
GREEN_SLOW_ZONE = 0.6           # line within this offset in near or middle -> slow down
FEEDFORWARD_FRACTION = 0.75     # fraction of the expected angle done fast, open-loop


def clamp(value, low, high):
    return max(low, min(high, value))

In [ ]:
def spin_test(seconds=2.15):
    """Spin in place for a fixed time; measure the angle turned and set ROTATION_TIME_360 = seconds*360/angle."""
    try:
        robot.set_motors(-GREEN_TURN_SPEED, GREEN_TURN_SPEED)
        time.sleep(seconds)
    finally:
        robot.stop()

# spin_test(2.15)   # uncomment to calibrate ROTATION_TIME_360 (the robot will spin)

## 3. Perception
`perceive(frame)` returns one `Observation` per frame: line offset in three bands (near / middle / far, −1 left … +1 right),
the green instruction (`LEFT`, `RIGHT`, `BOTH`, `CENTER`, `UNKNOWN`, `NONE`), the height of the lowest crossbar, and the
finish-marker bands. A green marker only counts when a crossbar is visible above it (rule 1.5.6: markers sit just before the junction).

In [ ]:
Observation = namedtuple("Observation",
                         "positions direction junction_y zone_markers markers crossing finish "
                         "junctions marker_junction_y junction_lines")
# junctions         : every crossbar in view, as y fractions where it meets the line, top -> bottom
# junction_y        : the lowest of them (None if none)
# marker_junction_y : the crossbar the valid green marker(s) belong to (None if no valid marker)
# junction_lines    : v16 - each crossbar as (y at image centre in px, slope); crossbars may be tilted

def detect_green(hsv, height):
    colour = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    kernel = np.ones((3, 3), dtype=np.uint8)
    colour = cv2.morphologyEx(colour, cv2.MORPH_OPEN, kernel)
    colour = cv2.morphologyEx(colour, cv2.MORPH_CLOSE, kernel)
    roi = colour.copy()
    roi[:int(height * GREEN_ROI_TOP), :] = 0
    roi[int(height * GREEN_ROI_BOTTOM):, :] = 0
    contours = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0 or not 0.4 <= w / float(h) <= 2.5:
            continue
        m = cv2.moments(contour)
        if m["m00"] <= 0:
            continue
        markers.append(dict(x=m["m10"] / m["m00"], y=m["m01"] / m["m00"], area=area, box=(x, y, w, h)))
    markers.sort(key=lambda item: item["area"], reverse=True)
    return markers, colour, roi

def through_line_x(strip):
    """Column of the vertical through-line in a band that also contains a horizontal stroke, or None."""
    black = strip > 0
    height, width = black.shape

    # 1. Straight vertical stroke: columns that are black over (almost) the whole band height.
    fill = black.mean(axis=0)
    cols = np.flatnonzero(fill >= STROKE_FILL)

    if cols.size:
        runs = np.split(
            cols, np.flatnonzero(np.diff(cols) > 1) + 1
        )
        return float(max(runs, key=len).mean())

    # 2. Slightly slanted through-line: match line ends at the top and bottom of the band.
    if height < 8:
        return None

    edge_rows = max(2, height // 8)
    max_line_width = 0.25 * width

    def edge_runs(edge):
        cols = np.flatnonzero(edge.mean(axis=0) >= 0.75)
        if cols.size == 0:
            return []

        runs = np.split(
            cols, np.flatnonzero(np.diff(cols) > 1) + 1
        )

        # A broad horizontal branch is not a through-line endpoint.
        return [
            r for r in runs
            if 2 <= len(r) <= max_line_width
        ]

    top_runs = edge_runs(black[:edge_rows])
    bottom_runs = edge_runs(black[-edge_rows:])

    if not top_runs or not bottom_runs:
        return None

    y_top = (edge_rows - 1) / 2.0
    y_bottom = height - 1 - y_top
    rows = np.arange(height)
    candidates = []

    for bottom in bottom_runs:
        for top in top_runs:
            x_bottom = float(bottom.mean())
            x_top = float(top.mean())

            slope = (x_bottom - x_top) / (y_bottom - y_top)

            # Only handle modest tilt here.
            # Strong bends remain with the existing corner logic.
            if abs(slope) > 0.8:
                continue

            xs = x_top + slope * (rows - y_top)
            supported = np.zeros(height, dtype=bool)

            # Small tolerance for image noise and tape edges.
            for offset in (-2, -1, 0, 1, 2):
                xi = np.rint(xs + offset).astype(int)
                valid = (xi >= 0) & (xi < width)
                supported[valid] |= black[rows[valid], xi[valid]]

            coverage = float(supported.mean())

            # Require a connected-looking path across almost all rows.
            if coverage < STROKE_FILL:
                continue

            midpoint = x_top + slope * (
                (height - 1) / 2.0 - y_top
            )
            candidates.append((coverage, midpoint))

    if not candidates:
        return None

    # Several plausible paths disagree: give up and let the caller use the centroid.
    centers = [candidate[1] for candidate in candidates]
    if max(centers) - min(centers) > 0.08 * width:
        return None

    return float(max(candidates, key=lambda item: item[0])[1])

def find_junctions(mask, height, width, ref_x):
    """Crossbars, corners and leftover line segments, including TILTED ones.

    v16: when the robot meets a junction at an angle (e.g. coming off a zig-zag) the crossbar is diagonal in
    the image, and no single row is mostly black. The mask is therefore also rotated by a few angles and the
    same row test is repeated (a projection at several angles). Each stroke becomes a straight line
    y = y0 + slope * (x - width/2) in the original image.
    Returns (lines, ys): lines as (y0 px, slope); ys = where each line crosses column ref_x, as fractions."""
    centre = (width / 2.0, height / 2.0)
    r0, r1 = int(CROSS_SEARCH[0] * height), int(CROSS_SEARCH[1] * height)
    lines = []
    for angle in JUNCTION_ANGLES:
        if angle == 0:
            rotated = mask
        else:
            M = cv2.getRotationMatrix2D(centre, angle, 1.0)
            rotated = cv2.warpAffine(mask, M, (width, height), flags=cv2.INTER_NEAREST)
        counts = (rotated[r0:r1] > 0).sum(axis=1)
        rows = np.flatnonzero(counts > CROSS_ROW_FRACTION * width)
        if rows.size == 0:
            continue
        max_rows = height if angle == 0 else CROSS_MAX_ROWS_TILTED * height
        for run in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1):
            if not CROSS_MIN_ROWS <= len(run) <= max_rows:
                continue
            yr = r0 + float(run.mean())
            if angle == 0:
                lines.append((yr, 0.0))
                continue
            # map the horizontal line y = yr of the rotated mask back into the original image
            inv = cv2.invertAffineTransform(M)
            (xa, ya), (xb, yb) = [inv.dot([x, yr, 1.0]) for x in (0.0, width - 1.0)]
            slope = (yb - ya) / (xb - xa)
            lines.append((ya + slope * (width / 2.0 - xa), slope))
    # the same stroke is usually found at neighbouring angles: keep one per position along the line
    lines.sort(key=lambda l: l[0] + l[1] * (ref_x - width / 2.0))
    merged = []
    for line in lines:
        y = line[0] + line[1] * (ref_x - width / 2.0)
        if merged and y - (merged[-1][0] + merged[-1][1] * (ref_x - width / 2.0)) < JUNCTION_MERGE * height:
            if abs(line[1]) < abs(merged[-1][1]):     # prefer the least-rotated estimate
                merged[-1] = line
            continue
        merged.append(line)
    ys = [(l[0] + l[1] * (ref_x - width / 2.0)) / height for l in merged]
    return merged, ys

def detect_line(frame, gray, green_colour):
    height, width = frame.shape[:2]
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    # dark green tape must never be mistaken for black tape
    mask[cv2.dilate(green_colour, np.ones((5, 5), np.uint8)) > 0] = 0
    overlay = frame.copy()
    positions, crossing = {}, set()
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            m = cv2.moments(contour)
            if 0.005 < fraction < 0.80 and m["m00"] > 0:
                cx = m["m10"] / m["m00"]
                if cv2.boundingRect(contour)[2] > CROSS_BAND_WIDTH * width:   # junction in this band
                    sx = through_line_x(strip)
                    if sx is not None:
                        cx = sx
                        crossing.add(name)
                positions[name] = (cx - width / 2.0) / (width / 2.0)
                cv2.circle(overlay, (int(cx), (y0 + y1) // 2), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
    return positions, crossing, mask, overlay

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ("far", "middle", "near") if positions.get(name) is not None]

def trace_approach_line(mask, width, height, start_x, top_row):
    """v16: follow the tape the robot is on, from the bottom of the view up to top_row.
    Returns {row: x}. At each row it keeps the narrow black run closest to the previous x, so it stays on
    the approach line and is not pulled onto a branch, unlike the band centroids."""
    path, x, misses = {}, float(start_x), 0
    max_width = GREEN_TRACE_MAX_WIDTH * width
    for row in range(int(GREEN_TRACE_START * height), int(top_row) - 1, -2):
        cols = np.flatnonzero(mask[row] > 0)
        best = None
        if cols.size:
            runs = [r for r in np.split(cols, np.flatnonzero(np.diff(cols) > 1) + 1) if len(r) <= max_width]
            if runs:
                best = min((float(r.mean()) for r in runs), key=lambda c: abs(c - x))
        if best is not None and (not path or abs(best - x) <= GREEN_TRACE_MAX_STEP):
            x = best
            path[row] = x
            misses = 0
        else:
            misses += 1
            if misses > GREEN_TRACE_MAX_MISSES:
                break
    return path

def crossbar_above(marker, line_x, junction_lines, width, height):
    """The crossbar a marker belongs to = the nearest stroke ABOVE it, measured at the approach line,
    if close enough. Returns its y (fraction) or None."""
    above = []
    for y0, slope in junction_lines:
        y = y0 + slope * (line_x - width / 2.0)          # where the crossbar meets the approach line
        y_at_marker = y0 + slope * (marker["x"] - width / 2.0)
        if y_at_marker < marker["y"]:
            above.append(y)
    if not above:
        return None
    nearest = max(above) / height
    return nearest if marker["y"] / height - nearest <= GREEN_JUNCTION_MAX_GAP else None

def classify_green_direction(positions, markers, width, height, mask, junction_lines):
    """Returns (direction, number of markers in the trigger zone, crossbar the valid markers belong to).

    v16: left/right is judged against the approach line traced at the marker's own height (perpendicular
    distance to that local line), not against the near->far band points, which jump onto a branch when the
    junction is seen at an angle."""
    zone = [m for m in markers if m["y"] >= height * GREEN_TRIGGER_Y]
    if not markers:
        return "NONE", 0, None
    if not zone:
        return "CENTER", 0, None
    # v16: seen at an angle, one marker of a dead-end pair reaches the zone before the other. Judge the
    # partner too, so the pair is read as BOTH instead of the first marker alone confirming LEFT/RIGHT.
    lowest_in_zone = min(m["y"] for m in zone)
    zone = [m for m in markers if m["y"] >= lowest_in_zone - height * GREEN_PAIR_Y_TOLERANCE]
    near = positions.get("near")
    start_x = width * (1.0 + near) / 2.0 if near is not None else width / 2.0
    path = trace_approach_line(mask, width, height, start_x, min(m["y"] for m in zone) - GREEN_TRACE_MARGIN)
    if len(path) < 3:
        return "UNKNOWN", len(zone), None
    sides = {"LEFT": [], "RIGHT": []}
    crossbars = []
    bottom = max(path)
    for marker in zone:
        rows = [r for r in path if abs(r - marker["y"]) <= GREEN_TRACE_MARGIN]
        if not rows:
            continue
        row = min(rows, key=lambda r: abs(r - marker["y"]))
        line_x = path[row]
        # local direction of the approach line: from the bottom of the trace up to the marker
        dx, dy = line_x - path[bottom], row - bottom
        if bottom - row < 10:
            dx, dy = 0.0, -1.0
        length = (dx * dx + dy * dy) ** 0.5
        dx, dy = dx / length, dy / length
        # Rule 1.5.6 - a valid marker sits just BEFORE a junction: its crossbar must be just above it.
        crossbar = crossbar_above(marker, line_x, junction_lines, width, height)
        if crossbar is None:
            continue
        score = (marker["x"] - line_x) * dy - (marker["y"] - row) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides["LEFT" if score > 0 else "RIGHT"].append(marker)
            crossbars.append(crossbar)
    marker_junction = max(crossbars) if crossbars else None
    if sides["LEFT"] and sides["RIGHT"]:
        paired = any(abs(a["y"] - b["y"]) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides["LEFT"] for b in sides["RIGHT"])
        return ("BOTH" if paired else "UNKNOWN"), len(zone), marker_junction
    if sides["LEFT"]:
        return "LEFT", len(zone), marker_junction
    if sides["RIGHT"]:
        return "RIGHT", len(zone), marker_junction
    return "CENTER", len(zone), None

def detect_finish_bands(frame):
    """Find short, broad red and bright neutral bands in the lower camera view.
    Returns (red_y, silver_y, silver_low_y) as fractions of image height, each None if not seen."""
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    y0, y1 = int(height * FINISH_ROI_TOP), int(height * FINISH_ROI_BOTTOM)
    roi = hsv[y0:y1]
    red1 = cv2.inRange(roi, np.array([0, 75, 45], np.uint8), np.array([12, 255, 255], np.uint8))
    red2 = cv2.inRange(roi, np.array([168, 75, 45], np.uint8), np.array([180, 255, 255], np.uint8))
    red = cv2.bitwise_or(red1, red2)
    silver = cv2.inRange(roi, np.array([0, 0, 135], np.uint8), np.array([180, 75, 255], np.uint8))
    # White floor can also be bright and neutral. Require a local brightness
    # change at the silver band's edge so a large uniform floor region is rejected.
    gray_roi = cv2.cvtColor(frame[y0:y1], cv2.COLOR_BGR2GRAY).astype(np.float32)
    row_mean = gray_roi.mean(axis=1)
    edge_change = np.zeros_like(row_mean)
    if len(row_mean) > 6:
        edge_change[3:-3] = np.maximum(
            np.abs(row_mean[3:-3] - row_mean[:-6]),
            np.abs(row_mean[3:-3] - row_mean[6:])
        )

    def broad_band(mask, min_coverage, require_edge=False):
        coverage = (mask > 0).mean(axis=1)
        eligible = coverage >= min_coverage
        if require_edge:
            eligible &= edge_change >= 5.0
        rows = np.flatnonzero(eligible)
        if rows.size < FINISH_ROW_MIN_PIXELS:
            return None
        runs = [r for r in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1)
                if FINISH_ROW_MIN_PIXELS <= len(r) <= int(0.22 * height)]
        if not runs:
            return None
        run = max(runs, key=len)
        lowest = max(runs, key=lambda r: r.max())
        return (y0 + float(run.mean())) / height, (y0 + float(lowest.max())) / height

    red_y = broad_band(red, FINISH_RED_MIN_COVERAGE)
    silver = broad_band(silver, FINISH_SILVER_MIN_COVERAGE, require_edge=True)
    # (red band centre, silver band centre, lowest silver edge = how close the foil is to the robot)
    return (None if red_y is None else red_y[0],
            None if silver is None else silver[0],
            None if silver is None else silver[1])

def perceive(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    markers, green_colour, green_roi = detect_green(hsv, height)
    positions, crossing, mask, overlay = detect_line(frame, gray, green_colour)
    near = positions.get("near")
    ref_x = width * (1.0 + near) / 2.0 if near is not None else width / 2.0
    junction_lines, junctions = find_junctions(mask, height, width, ref_x)
    junction_y = max(junctions) if junctions else None
    direction, zone, marker_junction_y = classify_green_direction(positions, markers, width, height,
                                                                  mask, junction_lines)
    finish = detect_finish_bands(frame)
    obs = Observation(positions, direction, junction_y, zone, markers, crossing, finish,
                      junctions, marker_junction_y, junction_lines)
    return obs, overlay, green_roi

In [ ]:
def synth(lines=(), markers=(), thickness=10):
    img = np.full((224, 224, 3), 235, np.uint8)
    for x0, y0, x1, y1 in lines:
        cv2.line(img, (x0, y0), (x1, y1), (20, 20, 20), thickness)
    for x, y in markers:
        cv2.rectangle(img, (x - 7, y - 7), (x + 7, y + 7), (40, 170, 40), -1)
    return img

def rotate(img, degrees):
    """Same drawing seen with the robot turned by `degrees` (positive = counter-clockwise in the image)."""
    M = cv2.getRotationMatrix2D((img.shape[1] / 2.0, img.shape[0] / 2.0), degrees, 1.0)
    return cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_NEAREST, borderValue=(235, 235, 235))

def self_test(verbose=True):
    V = (112, 0, 112, 223)
    cases = [
        ("straight line", synth([V]), lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is None),
        ("4-way: middle band not pulled sideways", synth([V, (0, 140, 223, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is not None),
        ("T with left branch: keep through-line", synth([V, (0, 140, 112, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05),
        ("unmarked left corner: still steer left", synth([(112, 223, 112, 140), (112, 140, 0, 140)]),
         lambda o: o.positions["middle"] is not None and o.positions["middle"] < -0.2),
        ("left marker before junction", synth([V, (0, 140, 223, 140)], [(94, 156)]),
         lambda o: o.direction == "LEFT"),
        ("right marker before junction", synth([V, (0, 140, 223, 140)], [(130, 156)]),
         lambda o: o.direction == "RIGHT"),
        ("dead end (both markers)", synth([V, (0, 140, 223, 140)], [(94, 156), (130, 156)]),
         lambda o: o.direction == "BOTH"),
        ("marker beyond the crossbar is ignored", synth([V, (0, 175, 223, 175)], [(94, 157)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        ("marker with no junction is ignored", synth([V], [(94, 170)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        # v15: right after a turn the line just left lies across the bottom of the image
        ("old line across bottom + left marker", synth([V, (0, 120, 223, 120), (0, 205, 223, 205)], [(94, 150)]),
         lambda o: o.direction == "LEFT" and abs(o.marker_junction_y - 0.54) < 0.05),
        ("sharp corner, then right marker", synth([(223, 205, 112, 205), (112, 205, 112, 0), (0, 120, 223, 120)], [(130, 150)]),
         lambda o: o.direction == "RIGHT"),
        ("beyond crossbar, far junction: ignored", synth([V, (0, 175, 223, 175), (0, 60, 223, 60)], [(94, 157)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
    ]
    # v16: junction seen at an angle (robot arriving off a zig-zag or not square to the line)
    cases += [
        ("diagonal crossbar, left marker", synth([(112, 223, 112, 130), (10, 190, 200, 80)], [(92, 160)]),
         lambda o: o.direction == "LEFT"),
        ("T seen at +25 deg, left marker", rotate(synth([V, (0, 120, 223, 120)], [(94, 150)]), 25),
         lambda o: o.direction == "LEFT"),
        ("T seen at -25 deg, right marker", rotate(synth([V, (0, 120, 223, 120)], [(130, 150)]), -25),
         lambda o: o.direction == "RIGHT"),
        ("dead end seen at +20 deg", rotate(synth([V, (0, 120, 223, 120)], [(94, 150), (130, 150)]), 20),
         lambda o: o.direction == "BOTH"),
        ("tilted line, no junction: no crossbar", rotate(synth([V]), 30),
         lambda o: o.junction_y is None),
        ("tilted line + marker, no junction: ignored", rotate(synth([V], [(94, 150)]), 30),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
    ]
    passed = 0
    for name, img, check in cases:
        obs = perceive(img)[0]
        ok = bool(check(obs))
        passed += ok
        if verbose:
            pos = {k: (None if v is None else round(v, 2)) for k, v in obs.positions.items()}
            jy = None if obs.junction_y is None else round(obs.junction_y, 2)
            print("%-42s %s  dir=%-7s junction=%s %s" % (name, "PASS" if ok else "FAIL", obs.direction, jy, pos))
    print("%d/%d passed" % (passed, len(cases)))
    return passed == len(cases)

self_test()

## 4. Controllers
`NavigationCoordinator` is the single decision point each frame:
finish > green maneuver > gap crossing > black line. Only one controller's command reaches the motors.
### 4a. BlackLineController

In [ ]:
class BlackLineController:
    """Follow black tape, pivot on sharp bends, and search briefly when the tape is lost.

    One step() per frame:  line lost? -> search  |  else compute error -> manage pivot -> follow.
    """
    def __init__(self):
        self.stopped = False
        self.last_direction = 0      # sign of the last clear steering error (+1 right, -1 left)
        self.lost_started = None     # time the line disappeared
        self.filtered_error = None   # smoothed error, ordinary following only
        # active pivot
        self.turn_started = None
        self.turn_direction = 0
        self.best_turn_error = None
        self.last_progress = None
        # pivot waiting for confirmation
        self.reset_candidate()

    # ------------------------------------------------------------ helpers
    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def reset_candidate(self):
        self.turn_candidate = 0
        self.turn_candidate_frames = 0
        self.turn_candidate_near_only = False

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = clamp(PIVOT_INNER_RATIO * speed, -MAX_SPEED, 0.0)
        return (speed, inner) if direction > 0 else (inner, speed)

    def start_turn(self, direction, now, error=1.0):
        self.turn_started = now
        self.turn_direction = direction
        self.best_turn_error = abs(error)
        self.last_progress = now

    # ------------------------------------------------------------ main step
    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            return self.search(now)

        returning = self.lost_started is not None
        self.lost_started = None

        error, zigzag = self.steering_error(positions, visible)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            failure = self.update_active_turn(error, near, middle, now, returning)
            if failure is not None:
                return failure

        if self.turn_started is None:
            error = self.update_turn_candidate(error, near, middle, now)

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        return self.follow(error, zigzag)

    # ------------------------------------------------------------ 1. line lost
    def search(self, now):
        self.reset_candidate()
        if self.lost_started is None:
            self.lost_started = now
        if not self.last_direction:
            return self.stop("line lost without a known turn direction")
        if now - self.lost_started >= SEARCH_TIMEOUT:
            return self.stop("line not reacquired")
        if self.turn_started is None:
            self.start_turn(self.last_direction, now)
        if now - self.turn_started >= TURN_HARD_TIMEOUT:
            return self.stop("turn timed out")
        left, right = self.pivot(self.turn_direction)
        return left, right, "SEARCH"

    # ------------------------------------------------------------ 2. steering error
    def steering_error(self, positions, visible):
        """Weighted band offset plus look-ahead. Returns (error in [-1, 1], zigzag flag)."""
        near, middle = positions.get("near"), positions.get("middle")
        farthest = positions.get("far")
        if farthest is None:
            farthest = middle

        # Near and middle already centred: ignore the far band so a side branch cannot pull the robot.
        protect_forward = (self.turn_started is None
                           and near is not None and abs(near) < LINE_CENTER_LIMIT
                           and (middle is None or abs(middle) < LINE_MIDDLE_CENTER_LIMIT))
        # Near and far on opposite sides: two bends in view, steer on the closer one only.
        zigzag = (near is not None and farthest is not None
                  and near * farthest < 0 and abs(farthest - near) >= ZIGZAG_SPREAD)

        used = [(n, v) for n, v in visible if n != "far"] if (protect_forward or zigzag) else visible
        error = sum(BAND_WEIGHTS[n] * v for n, v in used) / sum(BAND_WEIGHTS[n] for n, _ in used)

        if not protect_forward and not zigzag and near is not None and farthest is not None:
            error += LOOKAHEAD_GAIN * (farthest - near)
        return clamp(error, -1.0, 1.0), zigzag

    # ------------------------------------------------------------ 3a. pivot in progress
    def update_active_turn(self, error, near, middle, now, returning):
        """End the pivot when the line is centred; stop the robot if it stops making progress."""
        if returning:   # line reappeared after a search: restart progress tracking from here
            self.best_turn_error, self.last_progress = abs(error), now
        if near is not None and abs(near) < TURN_EXIT and (middle is None or abs(middle) < TURN_EXIT_MIDDLE):
            self.turn_started = None
            self.turn_direction = 0
            return None
        if abs(error) < self.best_turn_error - TURN_PROGRESS:
            self.best_turn_error, self.last_progress = abs(error), now
        if now - self.last_progress >= TURN_STALL_TIMEOUT:
            return self.stop("turn made no progress")
        if now - self.turn_started >= TURN_HARD_TIMEOUT:
            return self.stop("turn timed out")
        return None

    # ------------------------------------------------------------ 3b. should a pivot start?
    def update_turn_candidate(self, error, near, middle, now):
        """Start a pivot only after several frames of consistent evidence.
        Returns the error to steer with (limited while a large error is still unconfirmed)."""
        candidate, near_only, required = 0, False, LINE_TURN_CONFIRM_FRAMES
        if abs(error) >= TURN_ENTER:
            sign = 1 if error > 0 else -1
            near_and_middle_agree = (near is not None and middle is not None
                                     and abs(near) >= LINE_TURN_NEAR_MIN and abs(middle) >= LINE_TURN_MIDDLE_MIN
                                     and near * middle > 0 and near * sign > 0)
            near_alone = (near is not None and middle is None
                          and abs(near) >= LINE_TURN_NEAR_ONLY_MIN and near * sign > 0)
            if near_and_middle_agree:
                candidate = sign
            elif near_alone:
                candidate, near_only, required = sign, True, LINE_TURN_NEAR_ONLY_FRAMES

        if candidate == 0:
            self.reset_candidate()
        else:
            if candidate == self.turn_candidate and near_only == self.turn_candidate_near_only:
                self.turn_candidate_frames += 1
            else:
                self.turn_candidate, self.turn_candidate_frames, self.turn_candidate_near_only = candidate, 1, near_only
            if self.turn_candidate_frames >= required:
                self.start_turn(candidate, now, error)
                self.reset_candidate()

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            # Unconfirmed: keep driving and steer gently on the near band only.
            error = clamp(near if near is not None else 0.0, -LINE_PENDING_ERROR_LIMIT, LINE_PENDING_ERROR_LIMIT)
            self.filtered_error = error
        return error

    # ------------------------------------------------------------ 4. ordinary following
    def follow(self, error, zigzag):
        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        base = MIN_DRIVE_SPEED if zigzag else max(base, MIN_DRIVE_SPEED)
        steering_error = error
        if not zigzag and abs(error) < TURN_ENTER:
            if self.filtered_error is None:
                self.filtered_error = error
            else:
                self.filtered_error = (STEERING_FILTER_ALPHA * error
                                       + (1.0 - STEERING_FILTER_ALPHA) * self.filtered_error)
            steering_error = self.filtered_error
        else:
            self.filtered_error = error
        correction = KP * steering_error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, ("FOLLOW_SLOW" if zigzag else "FOLLOW")

### 4b. GapCrossingController

In [ ]:
class GapCrossingController:
    """Own gap evidence/crossing; delegate ordinary driving to BlackLineController."""
    def __init__(self):
        self.black_line = BlackLineController()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.black_line.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        values = [v for v in positions.values() if v is not None]

        if self.gap_started is not None:
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.black_line.stop("gap crossing timed out")
            self.confirm_frames = self.confirm_frames + 1 if (near is not None and middle is not None) else 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.black_line.lost_started = None
            self.black_line.last_direction = 0

        if not values:
            recent = self.straight_seen is not None and now - self.straight_seen <= GAP_APPROACH_MAX_AGE
            if self.black_line.turn_started is None and recent and self.straight_frames >= GAP_APPROACH_FRAMES:
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return self.black_line.step(positions, now)

        left, right, state = self.black_line.step(positions, now)
        aligned = max(abs(v) for v in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        return left, right, state

### 4c. GreenMarkerTurnController
`FOLLOW` → votes confirm a direction → `GREEN_ADVANCE` (keep rolling until the axle is over the junction) →
`GREEN_TURN` (ramp from forward into the in-place turn, rotate, finish when the line is centred) → `FOLLOW`.
A dead end (`BOTH`) skips the advance and turns around before the junction.

In [ ]:
GREEN_DIRECTIONS = ("LEFT", "RIGHT", "BOTH")

def line_centered(positions):
    return all(positions.get(n) is not None and abs(positions[n]) < TURN_EXIT for n in ("near", "middle"))


class GreenMarkerTurnController:
    """Confirm a green instruction and execute it. step() returns None while not in control.

    FOLLOW --(votes confirm LEFT/RIGHT)--> GREEN_ADVANCE --(axle over junction)--> GREEN_TURN --> FOLLOW
    FOLLOW --(votes confirm BOTH)------------------------------------------------> GREEN_TURN (U-turn)
    """
    def __init__(self):
        self.state = "FOLLOW"
        self.state_started = None
        self.direction = None
        self.votes = deque()          # (time, direction) of recent frames that showed a valid marker
        self.vote_junction_y = None   # crossbar the latest valid marker belongs to (handed to the advance)
        self.chain_votes = []         # v17: valid markers seen near the end of the current turn
        self.cooldown_until = 0.0
        self.approach_command = (BASE_SPEED, BASE_SPEED)
        self.ramp_from = self.approach_command
        self.stopped = False
        self.reason = ""
        self.gate = ""                # why a visible marker is not acted on yet (shown in the log)

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, "STOP: " + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    # ------------------------------------------------------------ main step
    def step(self, obs, now, can_start=True, last_command=(BASE_SPEED, BASE_SPEED)):
        if self.stopped:
            return self.stop(self.reason)
        self.gate = ""
        # A state may hand over to the next one within the same frame, hence plain ifs.
        if self.state == "FOLLOW":
            if not self.watch_for_marker(obs, now, can_start, last_command):
                return None
        if self.state == "GREEN_ADVANCE":
            command = self.advance_step(obs, now)
            if command is not None:
                return command
        if self.state == "GREEN_TURN":
            return self.turn_step(obs, now)
        return None

    # ------------------------------------------------------------ 1. confirm the marker
    def watch_for_marker(self, obs, now, can_start, last_command):
        """Collect votes; start a maneuver once they confirm a direction. Returns True if one started."""
        direction = obs.direction
        if direction in GREEN_DIRECTIONS and now >= self.cooldown_until:
            self.votes.append((now, direction))
            self.vote_junction_y = obs.marker_junction_y    # crossbar of the most recent valid marker
        while self.votes and now - self.votes[0][0] > GREEN_VOTE_WINDOW_TIME:
            self.votes.popleft()

        if now < self.cooldown_until:
            self.gate = "cooldown" if direction in GREEN_DIRECTIONS else ""
            return False
        if not can_start:
            self.gate = "blocked" if self.votes else ""
            return False
        decided = self.confirmed()
        if decided is None:
            self.gate = "votes %d/%d" % (len(self.votes), GREEN_VOTES_NEEDED) if self.votes else ""
            return False
        self.start_maneuver(decided, now, last_command)
        return True

    def start_maneuver(self, decided, now, last_command, chained=False):
        self.direction = decided
        self.votes.clear()
        self.both_votes = 0
        self.recheck_until = now + GREEN_RECHECK_TIME
        self.approach_command = tuple(last_command)
        if decided == "BOTH":
            self.begin_turn(now)          # dead end: turn around before the junction, no advance
        else:
            self.begin_advance(now, self.vote_junction_y, chained)   # keep rolling, no stop

    def confirmed(self, votes=None):
        counts = {d: 0 for d in GREEN_DIRECTIONS}
        for _, d in (self.votes if votes is None else votes):
            counts[d] += 1
        if counts["BOTH"] >= GREEN_VOTES_NEEDED:
            return "BOTH"
        left, right = counts["LEFT"], counts["RIGHT"]
        if max(left, right) >= GREEN_VOTES_NEEDED and left != right:
            return "LEFT" if left > right else "RIGHT"
        return None

    # ------------------------------------------------------------ 2. advance until the axle is over the junction
    def begin_advance(self, now, crossbar_y=None, chained=False):
        self.junction_seen = crossbar_y is not None
        self.last_junction_y = crossbar_y           # v15: the crossbar the marker belongs to
        self.reacquire = chained                    # v17: robot just rotated, crossbar may have moved
        self.passed_frames = 0
        self.post_started = None
        self.enter("GREEN_ADVANCE", now)

    def advance_step(self, obs, now):
        """Returns a drive command, or None once the turn has been started."""
        if now <= self.recheck_until and obs.direction == "BOTH":
            self.both_votes += 1
        if self.both_votes >= GREEN_BOTH_UPGRADE:      # the second marker showed up: U-turn instead
            self.direction = "BOTH"
            self.begin_turn(now)
            return None
        command = self.advance(obs, now)
        if command is not None:
            if not self.stopped:
                self.approach_command = command[:2]
            return command
        self.begin_turn(now)
        return None

    def advance(self, obs, now):
        """Closed-loop approach: drive until the crossbar has left the image bottom, then a short extra
        GREEN_POST_JUNCTION_TIME. Returns a command, or None when the axle is over the junction."""
        elapsed = now - self.state_started
        if elapsed >= GREEN_ADVANCE_TIMEOUT:
            return self.stop("junction not passed during advance")
        if self.post_started is None:
            jy = self.track_crossbar(obs)
            if jy is not None:
                self.junction_seen = True
                self.last_junction_y = jy
                under = jy >= JUNCTION_EXIT_Y
            elif self.junction_seen:
                under = self.last_junction_y >= JUNCTION_NEAR_Y
            else:
                under = elapsed >= GREEN_NO_JUNCTION_WAIT
            self.passed_frames = self.passed_frames + 1 if under else 0
            if self.passed_frames >= GREEN_PASS_FRAMES:
                self.post_started = now
        if self.post_started is not None and now - self.post_started >= GREEN_POST_JUNCTION_TIME:
            return None
        near = obs.positions.get("near")
        steer = clamp(KP * near, -GREEN_ADVANCE_STEER, GREEN_ADVANCE_STEER) if near is not None else 0.0
        speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
        label = "GREEN_ADVANCE_BLIND" if self.post_started is not None else "GREEN_ADVANCE"
        return clamp(speed + steer, 0.0, MAX_SPEED), clamp(speed - steer, 0.0, MAX_SPEED), label

    def track_crossbar(self, obs):
        """v15: position of the marker's crossbar in this frame, or None if it is no longer visible.
        Picks the stroke closest to where it was last frame, so a different stroke (the old line after
        a turn, or a farther junction) cannot be mistaken for it."""
        if self.last_junction_y is None:
            return obs.junction_y                  # nothing tracked yet: lowest stroke, as before
        last = self.last_junction_y
        if self.reacquire:   # first frame after a chained turn: search both ways, then track as usual
            low, high = last - JUNCTION_REACQUIRE, last + JUNCTION_REACQUIRE
            self.reacquire = False
        else:
            low, high = last - JUNCTION_TRACK_BACK, last + JUNCTION_TRACK_STEP
        near = [y for y in obs.junctions if low <= y <= high]
        return min(near, key=lambda y: abs(y - last)) if near else None

    # ------------------------------------------------------------ 3. ramp into the turn, rotate, reacquire
    def begin_turn(self, now):
        self.ramp_from = self.approach_command
        self.departed = False
        self.reacquire_frames = 0
        self.seen_ahead = False
        self.cross_frames = 0
        self.chain_votes = []
        self.enter("GREEN_TURN", now)

    def turn_command(self, positions, elapsed):
        expected = (2 if self.direction == "BOTH" else 1) * QUARTER_TURN_TIME
        near, middle = positions.get("near"), positions.get("middle")
        line_close = any(v is not None and abs(v) < GREEN_SLOW_ZONE for v in (near, middle))
        slow = elapsed >= FEEDFORWARD_FRACTION * expected and line_close
        speed = clamp(GREEN_TURN_SLOW_SPEED if slow else GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        suffix = "_SLOW" if slow else ""
        if self.direction == "RIGHT":
            return speed, -speed, "GREEN_TURN_RIGHT" + suffix
        label = "GREEN_U_TURN" if self.direction == "BOTH" else "GREEN_TURN_LEFT"
        return -speed, speed, label + suffix

    def line_crossed(self, positions):
        """v18: has the new line swept across the centre of the view?
        ahead > 0: line is still on the side the robot is turning towards (it moves toward the centre).
        The old line moves away from the centre on the other side, so it never counts as 'ahead'."""
        values = [v for v in (positions.get("near"), positions.get("middle")) if v is not None]
        if not values:
            return False
        offset = sum(values) / len(values)
        ahead = offset if self.direction == "RIGHT" else -offset    # LEFT and U-turn rotate left
        if ahead > GREEN_AHEAD_MIN:
            self.seen_ahead = True
        elif ahead < -GREEN_AHEAD_MIN:
            self.seen_ahead = False      # a line well past the centre is some other stroke: start over
        return self.seen_ahead and ahead <= GREEN_CROSS_TOL

    def turn_step(self, obs, now):
        positions = obs.positions
        elapsed = now - self.state_started
        centered = line_centered(positions)
        if not centered:
            self.departed = True

        # Smooth blend from the last forward command to the in-place turn (smoothstep).
        if elapsed < GREEN_TURN_RAMP_TIME:
            a = clamp(elapsed / GREEN_TURN_RAMP_TIME, 0.0, 1.0)
            blend = a * a * (3.0 - 2.0 * a)
            target_left, target_right, _ = self.turn_command(positions, 0.0)
            start_left, start_right = self.ramp_from
            return (start_left + blend * (target_left - start_left),
                    start_right + blend * (target_right - start_right), "GREEN_TURN_RAMP")

        turn_elapsed = elapsed - GREEN_TURN_RAMP_TIME
        uturn = self.direction == "BOTH"
        if turn_elapsed >= (UTURN_TIMEOUT if uturn else GREEN_TURN_TIMEOUT):
            return self.stop("green turn timed out")
        minimum = UTURN_MIN_TIME if uturn else GREEN_TURN_MIN_TIME
        # v17: remember valid markers for the NEXT junction seen late in this turn
        expected = (2 if uturn else 1) * QUARTER_TURN_TIME
        if turn_elapsed >= GREEN_CHAIN_START * expected and obs.direction in GREEN_DIRECTIONS:
            self.chain_votes.append((now, obs.direction))
            self.vote_junction_y = obs.marker_junction_y
        if self.chain_votes:
            self.gate = "chain %d/%d" % (len(self.chain_votes), GREEN_VOTES_NEEDED)
        if self.departed and turn_elapsed >= minimum and centered:
            self.reacquire_frames += 1
        else:
            self.reacquire_frames = 0
        crossed = self.line_crossed(positions)
        if self.departed and turn_elapsed >= minimum and crossed:
            self.cross_frames += 1
        else:
            self.cross_frames = 0
        if self.reacquire_frames < GREEN_REACQUIRE_FRAMES and self.cross_frames < GREEN_CROSS_FRAMES:
            return self.turn_command(positions, turn_elapsed)

        # v17: the next junction's marker was already confirmed during this turn: go straight into it.
        chained = self.confirmed(self.chain_votes)
        if chained is not None:
            self.chain_votes = []
            self.start_maneuver(chained, now, self.turn_command(positions, turn_elapsed)[:2], chained=True)
            if self.state == "GREEN_ADVANCE":
                command = self.advance_step(obs, now)
                if command is not None:
                    return command
            return self.turn_step(obs, now)

        # Done. The coordinator replaces this (0, 0) with a line-following command in the same frame.
        self.enter("FOLLOW", now)
        self.cooldown_until = now + GREEN_COOLDOWN
        self.votes.clear()
        return 0.0, 0.0, "GREEN_DONE"

### 4d. NavigationCoordinator

In [ ]:
class NavigationCoordinator:
    """Single decision point, highest priority first: finish > green maneuver > gap crossing > black line."""
    def __init__(self):
        self.green_turn = GreenMarkerTurnController()
        self.gap = GapCrossingController()
        self.stopped = False
        self.reason = ""
        self.active_controller = "BLACK_LINE"
        self.finish_red_seen_at = None
        self.finish_reached_at = None     # foil reached the near band; optional short roll before stopping
        self.last_command = (BASE_SPEED, BASE_SPEED)   # used to blend smoothly into a green turn

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        self.active_controller = "STOPPED"
        return 0.0, 0.0, "STOP: " + reason

    def step(self, obs, now):
        command = self.decide(obs, now)
        self.last_command = command[:2]
        return command

    def finish_detected(self, obs, now):
        """Finish = red band, then silver foil within FINISH_SEQUENCE_TIMEOUT, and the foil has reached the
        near band (v15: before, seeing it anywhere in the lower half of the image stopped the robot)."""
        red_y, silver_y, silver_low_y = obs.finish
        if red_y is not None:
            self.finish_red_seen_at = now
        if self.finish_red_seen_at is None:
            return False
        if now - self.finish_red_seen_at > FINISH_SEQUENCE_TIMEOUT:
            self.finish_red_seen_at = None
            return False
        return (silver_y is not None and (red_y is None or silver_y >= red_y - 0.05)
                and silver_low_y >= FINISH_STOP_Y)

    def decide(self, obs, now):
        if self.stopped:
            return self.stop(self.reason)

        if self.finish_reached_at is None and self.finish_detected(obs, now):
            self.finish_reached_at = now
        if self.finish_reached_at is not None:
            if now - self.finish_reached_at < FINISH_ROLL_TIME:
                self.active_controller = "FINISHING"
                return BASE_SPEED, BASE_SPEED, "FINISH_ROLL"
            self.stopped = True
            self.reason = "FINISH_LINE_DETECTED (red band followed by silver foil)"
            self.active_controller = "FINISHED"
            return 0.0, 0.0, self.reason

        # Green may pre-empt a black-line pivot only if a junction is visible (the crossbar caused it),
        # and never interrupts a gap crossing.
        line_turning = self.gap.black_line.turn_started is not None
        can_start_green = self.gap.gap_started is None and (not line_turning or obs.junction_y is not None)
        command = self.green_turn.step(obs, now, can_start=can_start_green, last_command=self.last_command)

        if command is not None:
            self.active_controller = "GREEN_TURN"
            if self.green_turn.stopped:
                return self.stop(self.green_turn.reason)
            if command[2] != "GREEN_DONE":
                return command
            # Turn finished: fresh line controller, and drive on in this same frame instead of sending (0, 0).
            self.gap = GapCrossingController()

        command = self.gap.step(obs.positions, now)
        self.active_controller = "GAP_CROSSING" if self.gap.gap_started is not None else "BLACK_LINE"
        if self.gap.black_line.stopped:
            return self.stop(command[2])
        return command

## 5. Preview and drive
`run_preview(seconds)` runs everything with the motors **off** and shows the overlay (line points, green boxes, red crossbar line)
and the green mask. `run_robot(seconds)` drives. Both print a summary at the end: loop rate, then one line per change of
controller / state / green gate. `run(40, drive=True, save_csv=True)` also writes the full per-frame log to a CSV.

In [ ]:
LOG_FIELDS = ("t", "proc_ms", "owner", "state", "marker", "green_gate",
              "near", "middle", "far", "junction_y", "left", "right")
LOG = deque(maxlen=6000)   # ~5 min at 20 Hz; memory stays bounded, no file I/O inside the loop

overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Ready for stationary preview")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))


def show_frame(frame, obs, overlay, green_mask, state, left, right, owner, gate, drive):
    """Draw the line points, green markers and crossbar on the overlay and update the widgets."""
    height, width = frame.shape[:2]
    points = get_line_points(obs.positions, width, height)
    for a, b in zip(points, points[1:]):
        cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
    for marker in obs.markers:
        x, y, w, h = marker["box"]
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
    for y0, slope in obs.junction_lines:            # every crossbar, drawn at its real tilt: thin red
        ya, yb = y0 - slope * width / 2.0, y0 + slope * width / 2.0
        cv2.line(overlay, (0, int(ya)), (width - 1, int(yb)), (0, 0, 255), 1)
    if obs.marker_junction_y is not None:           # where the marker's crossbar meets the line: yellow dot
        line_x = width * (1.0 + (obs.positions.get("near") or 0.0)) / 2.0
        cv2.circle(overlay, (int(line_x), int(obs.marker_junction_y * height)), 5, (0, 255, 255), -1)
    overlay_view.value = bgr8_to_jpeg(overlay)
    mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
    status_view.value = "%s | %s | marker=%s | gate=%s | L=%.2f R=%.2f | owner=%s" % (
        "DRIVE" if drive else "PREVIEW", state, obs.direction, gate or "-", left, right, owner)


def run(seconds=10, drive=False, show=True, save_csv=False):
    """Control loop. drive=False: preview only (motors never move). Motors always stop at the end."""
    robot.stop()
    nav = NavigationCoordinator()
    LOG.clear()
    start = time.monotonic()
    deadline = start + seconds
    previous_stamp = None
    next_display = 0.0
    display_period = 0.3 if drive else 0.1   # JPEG encoding is non-essential work; do less of it while driving
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame")
                break
            frame, captured_at = sample
            if time.monotonic() - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:          # wait for a new frame
                time.sleep(0.005)
                continue
            previous_stamp = captured_at

            t0 = time.monotonic()
            obs, overlay, green_mask = perceive(frame)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: deadline or slow image processing")
                break

            left, right, state = nav.step(obs, now)
            if drive:
                robot.set_motors(left, right)

            p = obs.positions
            gate = nav.green_turn.gate
            LOG.append((now - start, 1000.0 * (now - t0), nav.active_controller, state, obs.direction,
                        gate, p.get("near"), p.get("middle"), p.get("far"), obs.junction_y, left, right))
            if show and now >= next_display:
                show_frame(frame, obs, overlay, green_mask, state, left, right, nav.active_controller, gate, drive)
                next_display = now + display_period
            if nav.stopped:
                print("FINISH:" if nav.active_controller == "FINISHED" else "FAILURE:", state)
                break
            time.sleep(0.005)
    finally:
        robot.stop()
        print("Run ended; motors stopped.")
        summarize_log(save_csv)


def run_preview(seconds=20):
    run(seconds=seconds, drive=False)


def run_robot(seconds=2):
    run(seconds=seconds, drive=True)


def summarize_log(save_csv=False):
    """Loop timing, then one line per change of controller / state / green gate."""
    rows = list(LOG)
    if len(rows) < 2:
        print("No log data.")
        return
    t = np.array([r[0] for r in rows])
    proc = np.array([r[1] for r in rows])
    period = np.diff(t)
    print("frames %d | loop %.1f Hz (max period %.0f ms) | processing mean %.1f ms, max %.1f ms"
          % (len(rows), 1.0 / period.mean(), 1000 * period.max(), proc.mean(), proc.max()))
    previous = None
    for r in rows:
        key = (r[2], r[3].split(":")[0], r[5].split(" ")[0])
        if key != previous:
            print("  t=%6.2f  %-12s %-22s marker=%-7s gate=%-10s junction=%s"
                  % (r[0], r[2], r[3], r[4], r[5] or "-", "-" if r[9] is None else "%.2f" % r[9]))
            previous = key
    if save_csv:
        name = "run_%d.csv" % int(time.time())
        with open(name, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(LOG_FIELDS)
            writer.writerows(rows)
        print("Saved", name)

In [ ]:
run_preview(seconds=40)   # motors stay off

In [ ]:
run_robot(seconds=40)     # the robot moves when this cell runs

## 6. Cleanup

In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()